## Unimodal - Clinical Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Installation

!pip install -q monai torch torchvision torchaudio lifelines torchtuples pycox


In [ ]:
!pip install --upgrade scikit-learn scikit-survival skorch


### Model 1. Clinical data (Unimodal - Coxnet)

In [ ]:
df_clinical = pd.read_csv("/content/drive/MyDrive/NSCLC/df_clinical.csv")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler 
from lifelines.statistics import logrank_test
from lifelines import KaplanMeierFitter
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored, integrated_brier_score
from sksurv.linear_model import CoxnetSurvivalAnalysis


In [ ]:
# # Model 1 - Unimodal Coxnet


# Data Loading and Preparation

df_clinical_m1 = pd.read_csv("/content/drive/MyDrive/NSCLC/df_clinical.csv")

clinical_cols = [
    "age", "clinical.T.Stage", "Clinical.N.Stage",
    "Clinical.M.Stage", "Overall.Stage",
    "gender_male", "Histology_large cell",
    "Histology_nos", "Histology_squamous cell carcinoma"
]

X = df_clinical_m1[clinical_cols].values.astype("float32")
y_time = df_clinical_m1["Survival.time"].values.astype("float32")
y_event = df_clinical_m1["deadstatus.event"].values.astype("int")

# Stratified cross-validation

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

cindex_folds = []
ibs_folds = []
risk_all, time_all, event_all = [], [], []

param_grid = {
    'l1_ratio': [0.1, 0.5, 0.9],
    'alpha_min_ratio': [0.01, 0.05, 0.1]
}

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y_event), 1):
    print(f"\n===== Fold {fold}/{N_SPLITS} (Coxnet) =====")

    # Split
    X_train_full, X_test = X[train_idx], X[test_idx]
    y_time_train_full, y_time_test = y_time[train_idx], y_time[test_idx]
    y_event_train_full, y_event_test = y_event[train_idx], y_event[test_idx]

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    X_train_full = imputer.fit_transform(X_train_full)
    X_train_full = scaler.fit_transform(X_train_full)

    X_test = imputer.transform(X_test)
    X_test = scaler.transform(X_test)

 
    y_train_struct = Surv.from_arrays(event=y_event_train_full.astype(bool), time=y_time_train_full)
    y_test_struct  = Surv.from_arrays(event=y_event_test.astype(bool), time=y_time_test)

    # Base model

    model_base = CoxnetSurvivalAnalysis(l1_ratio=0.5, alpha_min_ratio=0.01, fit_baseline_model=True)

    # Grid Search
    gcv = GridSearchCV(model_base, param_grid, cv=3, n_jobs=-1)
    gcv.fit(X_train_full, y_train_struct)

    best_model = gcv.best_estimator_
    print(f"Best settings: {gcv.best_params_}")

    # Metrics

    risk_scores = best_model.predict(X_test)
    cindex = concordance_index_censored(y_event_test.astype(bool), y_time_test, risk_scores)[0]
    cindex_folds.append(cindex)

    # IBS Calculation

    surv_funcs = best_model.predict_survival_function(X_test)
    t_min = max(y_time_test.min(), y_time_train_full.min())
    t_max = min(y_time_test.max(), y_time_train_full.max())
    grid_fold = np.linspace(t_min + 1, t_max - 1, 100)

    preds_surv = np.vstack([fn(grid_fold) for fn in surv_funcs])

    try:
        ibs_val = integrated_brier_score(y_train_struct, y_test_struct, preds_surv, grid_fold)
        ibs_folds.append(ibs_val)
    except:
        ibs_val = np.nan

    print(f"C-Index: {cindex:.4f} | IBS: {ibs_val:.4f}")

    risk_all.append(risk_scores)
    time_all.append(y_time_test)
    event_all.append(y_event_test)

# Results and Plotting

def get_stats(data):
    arr = np.array([x for x in data if not np.isnan(x)])
    mean = np.mean(arr)
    std = np.std(arr, ddof=1)
    sem = stats.sem(arr)
    ci = stats.t.interval(0.95, len(arr)-1, loc=mean, scale=sem)
    return mean, std, ci

c_m, c_s, c_ci = get_stats(cindex_folds)
i_m, i_s, i_ci = get_stats(ibs_folds)

print("\n" + "="*60)
print("FINAL RESULTS - MODEL 1 (COXNET)")
print(f"C-Index: {c_m:.4f} ± {c_s:.4f} | IC 95%: [{c_ci[0]:.4f}, {c_ci[1]:.4f}]")
print(f"IBS:     {i_m:.4f} ± {i_s:.4f} | IC 95%: [{i_ci[0]:.4f}, {i_ci[1]:.4f}]")
print("="*60)

risks, times, events = np.concatenate(risk_all), np.concatenate(time_all), np.concatenate(event_all)
threshold = np.median(risks)
high, low = (risks >= threshold), (risks < threshold)
p_value = logrank_test(times[high], times[low], events[high], events[low]).p_value

plt.figure(figsize=(8, 6))
kmf = KaplanMeierFitter()
kmf.fit(times[high]/365.25, events[high], label="High Risk")
kmf.plot_survival_function()
kmf.fit(times[low]/365.25, events[low], label="Low Risk")
kmf.plot_survival_function()

plt.text(0.60, 0.35, f"p = {p_value:.4f}", transform=plt.gca().transAxes,
         fontsize=10, bbox=dict(boxstyle="round", facecolor="white", alpha=0.6))


plt.xlabel("Time (years)"); plt.ylabel("Survival Probability"); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("unimodal_model_1.png", dpi=300)
plt.show()

### Model 2. Clinical data (Unimodal - Random Survival Forest (RSF))

In [ ]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer

from lifelines.statistics import logrank_test
from lifelines import KaplanMeierFitter

from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored, integrated_brier_score

In [ ]:

# Output Settings

SAVE_DIR = "/content/drive/MyDrive/NSCLC/Resultados_Modelo_2/"
os.makedirs(SAVE_DIR, exist_ok=True)

# Data Loading and Preparation

df_clinical = pd.read_csv("/content/drive/MyDrive/NSCLC/df_clinical.csv")

clinical_cols = [
    "age", "clinical.T.Stage", "Clinical.N.Stage",
    "Clinical.M.Stage", "Overall.Stage",
    "gender_male", "Histology_large cell",
    "Histology_nos", "Histology_squamous cell carcinoma"
]

X = df_clinical[clinical_cols].values.astype("float32")
y_time = df_clinical["Survival.time"].values.astype("float32")
y_event = df_clinical["deadstatus.event"].values.astype("int")

# Cross-validation with grid search

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

cindex_folds = []
ibs_folds = []
risk_all, time_all, event_all = [], [], []

param_grid = {
    'n_estimators': [200, 300],            
    'max_features': ['sqrt', None],        
    'min_samples_leaf': [5, 10, 15],       
    'min_samples_split': [10, 20],         
    'max_depth': [None, 7, 10]            
}

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y_event), 1):
    print(f"\n===== Fold {fold}/{N_SPLITS} ====")

    X_train_full, X_test = X[train_idx], X[test_idx]
    y_time_train_full, y_time_test = y_time[train_idx], y_time[test_idx]
    y_event_train_full, y_event_test = y_event[train_idx], y_event[test_idx]

    imputer = SimpleImputer(strategy="median")
    X_train_full = imputer.fit_transform(X_train_full)
    X_test = imputer.transform(X_test)

    y_train_struct = Surv.from_arrays(event=y_event_train_full.astype(bool), time=y_time_train_full)
    y_test_struct  = Surv.from_arrays(event=y_event_test.astype(bool), time=y_time_test)

    # Initializes the base RSF model
    rsf_base = RandomSurvivalForest(n_jobs=-1, random_state=42)

    
    gcv = GridSearchCV(rsf_base, param_grid, cv=3, n_jobs=-1, error_score='raise')
    gcv.fit(X_train_full, y_train_struct)

    best_model = gcv.best_estimator_
    print(f"Melhores parâmetros no Fold {fold}: {gcv.best_params_}")

    # C-INDEX
    risk_scores = best_model.predict(X_test)
    cindex = concordance_index_censored(y_event_test.astype(bool), y_time_test, risk_scores)[0]
    cindex_folds.append(cindex)

    # IBS
    surv_funcs = best_model.predict_survival_function(X_test)
    t_min = max(y_time_test.min(), y_time_train_full.min())
    t_max = min(y_time_test.max(), y_time_train_full.max())
    grid_fold = np.linspace(t_min + 1, t_max - 1, 100)

    preds_surv = np.vstack([fn(grid_fold) for fn in surv_funcs])

    try:
        ibs_val = integrated_brier_score(y_train_struct, y_test_struct, preds_surv, grid_fold)
        ibs_folds.append(ibs_val)
    except:
        ibs_val = np.nan # Caso o grid falhe

    print(f"C-Index: {cindex:.4f} | IBS: {ibs_val:.4f}")

    risk_all.append(risk_scores)
    time_all.append(y_time_test)
    event_all.append(y_event_test)

# Final Statistical Report

def get_stats(data):
    arr = np.array([x for x in data if not np.isnan(x)])
    mean = np.mean(arr)
    std = np.std(arr, ddof=1)
    sem = stats.sem(arr)
    ci = stats.t.interval(0.95, len(arr)-1, loc=mean, scale=sem)
    return mean, std, ci

c_m, c_s, c_ci = get_stats(cindex_folds)
i_m, i_s, i_ci = get_stats(ibs_folds)

risks, times, events = np.concatenate(risk_all), np.concatenate(time_all), np.concatenate(event_all)
threshold = np.median(risks)
high, low = (risks >= threshold), (risks < threshold)
p_value = logrank_test(times[high], times[low], events[high], events[low]).p_value

print("\n" + "="*60)
print(f"C-Index: {c_m:.4f} ± {c_s:.4f} | IC 95%: [{c_ci[0]:.4f}, {c_ci[1]:.4f}]")
print(f"IBS:     {i_m:.4f} ± {i_s:.4f} | IC 95%: [{i_ci[0]:.4f}, {i_ci[1]:.4f}]")
print(f"p-value: {p_value:.6f}")
print("="*60)


df_results = pd.DataFrame({
    "fold": range(1, N_SPLITS+1),
    "cindex": cindex_folds,
    "ibs": ibs_folds
})
df_results.to_csv(os.path.join(SAVE_DIR, "metrics_per_fold_model2.csv"), index=False)

# JSON 
import json 
summary = {
    "cindex_mean": float(c_m),
    "cindex_std": float(c_s),
    "cindex_ci_low": float(c_ci[0]),
    "cindex_ci_high": float(c_ci[1]),
    "ibs_mean": float(i_m),
    "ibs_std": float(i_s),
    "ibs_ci_low": float(i_ci[0]),
    "ibs_ci_high": float(i_ci[1]),
    "logrank_p_value": float(p_value)
}

with open(os.path.join(SAVE_DIR, "summary_model2.json"), "w") as f:
    json.dump(summary, f, indent=4)



plt.figure(figsize=(8, 6))
kmf = KaplanMeierFitter()
kmf.fit(times[high]/365.25, events[high], label=f"High Risk (n={sum(high)})") 
kmf.plot_survival_function(ci_show=True)
kmf.fit(times[low]/365.25, events[low], label=f"Low Risk (n={sum(low)})")
kmf.plot_survival_function(ci_show=True)

#plt.text(0.60, 0.35, f"p = {p_value:.4f}", transform=plt.gca().transAxes,
#         fontsize=10, bbox=dict(boxstyle="round", facecolor="white", alpha=0.6))

plt.xlabel("Time (years)"); plt.ylabel("Survival Probability"); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "km_model_2.png"), dpi=300)
plt.show()